In [15]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langgraph.checkpoint.memory import InMemorySaver  # inmemorysaver is a type of checkpointer which saves intermediate and final state values in RAM memory, used only for demos not production ready 

In [16]:
load_dotenv()

model = ChatGroq(
    api_key=os.getenv("GROQ_API_KEY"),
    model="llama-3.1-8b-instant"
)

In [17]:
class JokeState(TypedDict):
    topic:str
    joke:str
    explanation:str

In [18]:
def generate_joke(state:JokeState):
    prompt = f"Generate a joke on topic: {state['topic']}"
    response=model.invoke(prompt).content
    state['joke'] = response
    return state

In [19]:
def generate_explanation(state:JokeState):
    prompt = f"Explain the joke: {state['joke']}"
    response=model.invoke(prompt).content
    state['explanation'] = response
    return state

In [20]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()  # create an instance of the checkpointer

workflow = graph.compile(checkpointer=checkpointer)

In [21]:
config1={"configurable":{"thread_id":"1"}}
workflow.invoke({'topic':'pizza'}, config=config1)

{'topic': 'pizza',
 'joke': 'Why was the pizza in a bad mood? \n\nBecause it was feeling a little crusty.',
 'explanation': 'The joke is a play on words. The phrase "feeling a little crusty" typically means feeling irritable or short-tempered. However, in this joke, "crusty" also refers to the crust of a pizza, which is the outer layer of the pizza. The pun creates a humorous connection between the pizza\'s bad mood and its literal crust.'}

In [22]:
workflow.get_state(config1)  # for getting the final state value

StateSnapshot(values={'topic': 'pizza', 'joke': 'Why was the pizza in a bad mood? \n\nBecause it was feeling a little crusty.', 'explanation': 'The joke is a play on words. The phrase "feeling a little crusty" typically means feeling irritable or short-tempered. However, in this joke, "crusty" also refers to the crust of a pizza, which is the outer layer of the pizza. The pun creates a humorous connection between the pizza\'s bad mood and its literal crust.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f13b21d-a43e-6552-8002-233babea1c09'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-04-18T12:26:40.703802+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f13b21d-a162-6608-8001-291d2556b6dd'}}, tasks=(), interrupts=())

In [23]:
list(workflow.get_state_history(config1))  # for getting the inermediate state values
# why 4 values ? one for initial state, one after generating joke, one after generating explanation and one for final state 
# start has no value so state={} -> see last one 

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why was the pizza in a bad mood? \n\nBecause it was feeling a little crusty.', 'explanation': 'The joke is a play on words. The phrase "feeling a little crusty" typically means feeling irritable or short-tempered. However, in this joke, "crusty" also refers to the crust of a pizza, which is the outer layer of the pizza. The pun creates a humorous connection between the pizza\'s bad mood and its literal crust.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f13b21d-a43e-6552-8002-233babea1c09'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-04-18T12:26:40.703802+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f13b21d-a162-6608-8001-291d2556b6dd'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'topic': 'pizza', 'joke': 'Why was the pizza in a bad mood? \n\nBecause it was feeling a little crusty.'}, next=('generat

In [24]:
config2={"configurable":{"thread_id":"2"}}
workflow.invoke({'topic':'pasta'}, config=config2)

{'topic': 'pasta',
 'joke': 'Why did the pasta go to therapy? \n\nBecause it was feeling a little "drained."',
 'explanation': 'The joke is a play on words. "Feeling drained" is a common phrase used to describe someone who is emotionally or mentally exhausted. However, in the context of pasta, "drained" also has a literal meaning, referring to the process of removing excess water from cooked pasta. The pun relies on the dual meaning of "drained" to create a humorous connection between the setup (pasta going to therapy) and the punchline (feeling a little "drained").'}

In [25]:
workflow.get_state(config2)  # for getting the final state value

StateSnapshot(values={'topic': 'pasta', 'joke': 'Why did the pasta go to therapy? \n\nBecause it was feeling a little "drained."', 'explanation': 'The joke is a play on words. "Feeling drained" is a common phrase used to describe someone who is emotionally or mentally exhausted. However, in the context of pasta, "drained" also has a literal meaning, referring to the process of removing excess water from cooked pasta. The pun relies on the dual meaning of "drained" to create a humorous connection between the setup (pasta going to therapy) and the punchline (feeling a little "drained").'}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f13b21d-a985-68da-8002-0bc749a9feac'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-04-18T12:26:41.257265+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f13b21d-a683-6556-8001-1fa20b95a295'}}, tasks=(), interrupts=())

In [26]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'pasta', 'joke': 'Why did the pasta go to therapy? \n\nBecause it was feeling a little "drained."', 'explanation': 'The joke is a play on words. "Feeling drained" is a common phrase used to describe someone who is emotionally or mentally exhausted. However, in the context of pasta, "drained" also has a literal meaning, referring to the process of removing excess water from cooked pasta. The pun relies on the dual meaning of "drained" to create a humorous connection between the setup (pasta going to therapy) and the punchline (feeling a little "drained").'}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f13b21d-a985-68da-8002-0bc749a9feac'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-04-18T12:26:41.257265+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f13b21d-a683-6556-8001-1fa20b95a295'}}, tasks=(), interrupts=()),
 StateSnapshot